# Algorithm 3.2 — Kepler's equation for the hyperbola

**Goal:** the open-orbit twin of 3.1. Given an eccentricity $e>1$ and the hyperbolic mean anomaly $M$ (still a clock), solve **Kepler's equation for the hyperbola** $\;M = e\sinh F - F\;$ for the hyperbolic eccentric anomaly $F$.

**Code (answer key):** [`kepler_H`](../curtis/algorithms/alg_3_2_kepler_H.py) · **Book:** §3.7, Algorithm 3.2 · **Worked example:** 3.5

Everything mirrors 3.1 — same Newton's method, same role for each symbol — with one swap: $\sin/\cos \to \sinh/\cosh$.

## Read first

| Symbol | Name | Meaning |
|---|---|---|
| $M$ | hyperbolic mean anomaly | the clock: $M = \dfrac{\mu^2}{h^3}\,(e^2-1)^{3/2}\,(t-t_p)$ (dimensionless) |
| $F$ | hyperbolic eccentric anomaly | the unknown we solve for (argument of $\sinh$) |
| $\theta$ | true anomaly | the real angle from perigee |
| $e$ | eccentricity | $e>1$ here (hyperbola) |

**Kepler's equation (hyperbola):** $\;M = e\sinh F - F\;$ (Eq 3.40). Transcendental again — solved with Newton's method.

## The picture (an open orbit)

When $e>1$ the orbit is a **hyperbola**: the satellite sweeps past the focus once and leaves forever — a fly-by or an escape trajectory. There is no period and no apogee; instead the true anomaly has a hard limit $\theta_\infty=\cos^{-1}(-1/e)$ where $r\to\infty$ (the asymptote direction).

The bookkeeping is identical to the ellipse, with the circular functions replaced by hyperbolic ones:

| | ellipse ($e<1$) | hyperbola ($e>1$) |
|---|---|---|
| anomaly | $E$ | $F$ |
| Kepler's eqn | $M = E - e\sin E$ | $M = e\sinh F - F$ |
| Newton denominator | $1 - e\cos E$ | $e\cosh F - 1$ |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Arc

In [ ]:
# A hyperbolic trajectory (illustrative: e=1.5).
e_ill = 1.5
p = 1.0                                   # semi-latus rectum (units)
th_inf = np.arccos(-1/e_ill)              # true-anomaly limit (asymptote)
th = np.linspace(-0.86*th_inf, 0.86*th_inf, 400)
r = p / (1 + e_ill*np.cos(th))
x, y = r*np.cos(th), r*np.sin(th)

F0 = np.array([0.0, 0.0])                 # focus (Earth)
peri = np.array([p/(1 + e_ill), 0.0])     # perigee (theta = 0)
th_s = np.radians(75)                     # an illustrative satellite position
rs = p/(1 + e_ill*np.cos(th_s))
P = np.array([rs*np.cos(th_s), rs*np.sin(th_s)])

fig, ax = plt.subplots(figsize=(6.6, 6.2))
ax.plot(x, y, color='#3d3d3a', lw=2)                                  # the hyperbola
for s in (+1, -1):                                                    # asymptote directions
    ax.plot([0, 2.8*np.cos(s*th_inf)], [0, 2.8*np.sin(s*th_inf)],
            '--', color='#9c9a92', lw=1)
ax.plot([0, P[0]], [0, P[1]], color='#D4537E', lw=1.6)                # r to satellite
ax.add_patch(Arc(F0, 0.6, 0.6, theta1=0, theta2=75, color='#D4537E', lw=2))

for pt, col in [(F0, '#3B8BD4'), (peri, '#3d3d3a'), (P, '#D4537E')]:
    ax.plot(*pt, 'o', color=col, ms=6)
ax.annotate('focus (Earth)', F0 + [-0.07, -0.24], color='#3B8BD4', ha='right')
ax.annotate('perigee', peri + [0.12, -0.04], color='#3d3d3a', ha='left')
ax.annotate('satellite', P + [0.10, 0.04], color='#D4537E', ha='left')
ax.annotate(r'$\theta$', F0 + [0.42, 0.20], color='#D4537E', fontsize=15)
ax.annotate(r'asymptote ($\theta\!\to\!\theta_\infty$)',
            [2.8*np.cos(th_inf), 2.8*np.sin(th_inf)] + np.array([-0.2, 0.12]),
            color='#73726c', fontsize=10, ha='center')
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('A hyperbolic orbit (e > 1): an open fly-by')
plt.show()

## See it — F is the hyperbola's eccentric anomaly

The figure shows the open orbit: Earth at the focus, the satellite at true anomaly $\theta$, and the dashed **asymptotes** it approaches as $\theta\to\theta_\infty$ and $r\to\infty$.

- There is **no auxiliary circle** here — a hyperbola has no neat circular parent. $F$ is defined *algebraically*, as the hyperbolic-function analogue of $E$: where the ellipse used $x=a\cos E$, the hyperbola uses $x = a\cosh F$ (with $a<0$). It is the number that plays $E$'s role.
- $M$ is still the **clock** — it grows linearly with time — but it is dimensionless and unbounded (no period to wrap around).
- Solve $M\to F$ with Newton, then $F\to\theta$ with a closed-form formula, exactly as in 3.1.

## Where these equations come from

### Why $M = e\sinh F - F$
The derivation parallels the ellipse. Kepler's second law still gives "area swept $\propto$ time," but the area under a hyperbola is evaluated with **hyperbolic** functions instead of circular ones (the substitution $x=a\cosh F$ in place of $x=a\cos E$). Carrying the integral through yields
$$\boxed{\,M = e\sinh F - F\,}\qquad(3.40),$$
with $M=\dfrac{\mu^2}{h^3}(e^2-1)^{3/2}(t-t_p)$ — linear in time, so still a uniform clock.

### Why Newton's method (and the exact update)
Set $f(F)=e\sinh F - F - M$ and find its root. Then $f'(F)=e\cosh F - 1$, so
$$F_{k+1} = F_k - \frac{e\sinh F_k - F_k - M}{e\cosh F_k - 1}.$$
The denominator $e\cosh F - 1 \ge e - 1 > 0$ for $e>1$, so it never blows up, and convergence is quadratic. A simple seed $F_0 = M$ works (the book's choice).

### From $F$ to $\theta$
$$\tan\frac{\theta}{2} = \sqrt{\frac{e+1}{e-1}}\,\tanh\frac{F}{2}\qquad(3.41b).$$

## Step by step (in code order)

**1. Starting guess.** $\;F = M$.

**2. Newton iteration** until the correction `ratio` is below the tolerance:
$$\text{ratio} = \frac{e\sinh F - F - M}{e\cosh F - 1},\qquad F \leftarrow F - \text{ratio}.$$
Repeat `while abs(ratio) > tol`.

**↓ Now type your implementation below.** It is the same `while`-loop shape as 3.1 — just swap $\sin\to\sinh$, $\cos\to\cosh$, and mind the sign order ($e\sinh F - F$, and $e\cosh F - 1$). Answer key linked above; peek only after you try.

In [ ]:
def kepler_H(e, M, tol=1.0e-8):
    """Hyperbolic eccentric anomaly F solving M = e sinh F - F  (e > 1)."""

    # 1. Starting guess:  F = M

    # 2. Newton iteration until |ratio| < tol:
    #        ratio = (e*sinh(F) - F - M) / (e*cosh(F) - 1)
    #        F = F - ratio

    # return F
    raise NotImplementedError("fill in steps 1-2, then delete this line")

## Verify — Example 3.5

**Input:** $e = 2.7696$, $\;M = 40.69$.

**Expected:** $F = 3.46309$.

Run the cell below once your function is typed.

In [ ]:
e = 2.7696
M = 40.69
F = kepler_H(e, M)

# true anomaly from F (Eq 3.41b), just to see the geometry
theta = 2*np.arctan(np.sqrt((e + 1)/(e - 1))*np.tanh(F/2))

print(f"F     = {F:.6g}        (expect 3.46309)")
print(f"theta = {np.degrees(theta):.5g} deg")
print(f"check : e*sinh(F) - F = {e*np.sinh(F) - F:.6g}  (should equal M = {M})")

assert abs(F - 3.46309) < 1e-4
assert abs((e*np.sinh(F) - F) - M) < 1e-6
print("\nAll checks passed ✔")

## What this confirms

- A **hyperbola handles exactly like an ellipse** for time-of-flight — same Newton loop, just $\sin/\cos\to\sinh/\cosh$. The two cases are one idea.
- $M$ stays the clock; $F$ is the solved-for anomaly; $\theta$ is the geometry — same trio as 3.1.
- Open orbits ($e>1$) have no period: $M$ runs off to infinity and $\theta$ stops at $\theta_\infty=\cos^{-1}(-1/e)$.

**Next:** the **Stumpff functions** $C(z), S(z)$ — a small building block that lets one *universal* formulation (Algorithm 3.3) cover ellipse, parabola **and** hyperbola at once, instead of the separate 3.1 / 3.2 we have now.